In [1]:
# import os
# import random #데이터 샘플링
# from collections import Counter # count 용도

import numpy as np
import pandas as pd

# from geopy import distance # 거리 계산
# import geopy.distance
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

# 시각화
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
plt.style.use('fivethirtyeight')

# 한글, 마이너스 깨짐 방지
from matplotlib import rc, font_manager, rcParams
font=font_manager.FontProperties(fname="c:/Windows/Fonts/malgun.ttf").get_name()
rc('font', family = font)
rcParams['axes.unicode_minus'] = False

## 데이터 확인하기

In [2]:
df = pd.read_csv('./dataset/2019 관악구 업무추진비.csv')
df.head()

,연번,사용자,사용일시,사용시간,사용금액,사용장소,주소,사용방법,사용내역,집행대상
0,1,의장,2019-01-02,08:44,58500,완산정,서울 관악구 봉천로 484,신용카드,의정활동 및 의회운영 관련 업무휴대경비,의장 등 9명
1,2,의장,2019-01-02,12:36,89000,제주은갈치,서울 관악구 관악로 139,신용카드,의정활동 및 의회운영 관련 업무휴대경비,의장 등 7명
2,3,의장,2019-01-04,12:53,14000,아리차이,서울 관악구 신림동길 4,신용카드,의정활동 및 의회운영 관련 업무휴대경비,의장 등 2명
3,4,의장,2019-01-09,12:33,77000,남원추어탕,서울 관악구 쑥고개로 135,신용카드,의정활동 및 의회운영 관련 업무휴대경비,의장 등 7명
4,5,의장,2019-01-09,14:48,83300,카페베네,서울 관악구 남부순환로 2082-29,신용카드,의정활동 및 의회운영 관련 업무휴대경비,의장 등 17명


In [3]:
df.tail()

,연번,사용자,사용일시,사용시간,사용금액,사용장소,주소,사용방법,사용내역,집행대상
1999,101,의회사무국,2019-12-27,20:27,132000,시골집,서울 관악구 낙성대로 22-7,신용카드,의정업무 추진관계자 간담회,사무국장 등 10명
2000,102,의회사무국,2019-12-30,13:00,220000,만리장성,서울 관악구 관악로 146,신용카드,사무국 직원 격려,의정팀장 등 10명
2001,103,의회사무국,2019-12-30,13:08,87000,쌈마을,서울 관악구 남부순환로226길 31,신용카드,의사팀 업무추진 관련 간담회비,의사팀 직원 10명
2002,104,의회사무국,2019-12-30,14:01,174000,장보고마트,서울 관악구 남부순환로230길,신용카드,부서운영 물품(음료) 구입,"의원, 직원, 방문민원인 등"
2003,105,의회사무국,2019-12-30,20:24,123000,제주은갈치,서울 관악구 관악로 139,신용카드,신년인사회 추진 관련 관계자 간담회,의정팀장 등 7명


In [4]:
df.shape

(2004, 10)

In [5]:
df.columns

Index(['연번', '사용자', '사용일시', '사용시간', '사용금액', '사용장소', '주소', '사용방법', '사용내역',
       '집행대상'],
      dtype='object')

In [6]:
df.isnull().sum()

연번        0
사용자       0
사용일시      0
사용시간    452
사용금액      0
사용장소     11
주소       11
사용방법      0
사용내역      0
집행대상     28
dtype: int64

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2004 entries, 0 to 2003
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   연번      2004 non-null   int64 
 1   사용자     2004 non-null   object
 2   사용일시    2004 non-null   object
 3   사용시간    1552 non-null   object
 4   사용금액    2004 non-null   int64 
 5   사용장소    1993 non-null   object
 6   주소      1993 non-null   object
 7   사용방법    2004 non-null   object
 8   사용내역    2004 non-null   object
 9   집행대상    1976 non-null   object
dtypes: int64(2), object(8)
memory usage: 156.7+ KB


In [8]:
df.sum()

연번                                                  88964
사용자     의장의장의장의장의장의장의장의장의장의장의장의장의장의장의장의장의장의장의장의장의장의장의장...
사용일시    2019-01-022019-01-022019-01-042019-01-092019-0...
사용금액                                            224074380
사용방법    신용카드신용카드신용카드신용카드신용카드신용카드신용카드신용카드현금신용카드신용카드신용카드...
사용내역    의정활동 및 의회운영 관련 업무휴대경비의정활동 및 의회운영 관련 업무휴대경비의정활동...
dtype: object

In [9]:
df['사용장소'].nunique()

611

In [10]:
df['사용내역'].nunique()

213

+ 총 사용금액 224,074,380 (2억2천~)
+ 장소 611곳

+ 필요한 것
    - 필요없는 컬럼 정리
    - 컬럼 타입 변경
    - 음식점인지 아닌지 구별하기
    - 음식 카테고리 추가하기

## 데이터 전처리하기

+ 필요없는 컬럼 삭제
+ 컬럼 데이터 타입 변경

In [11]:
# 사용방법 컬럼 확인
df['사용방법'].unique()

array(['신용카드', '현금', '제로페이'], dtype=object)

In [12]:
# 사용방법 컬럼 확인
df['사용자'].unique()

array(['의장', '부의장', '의회운영위원장', '행정재경위원장', '보건복지위원장', '도시건설위원장', '관악구의회',
       '의회사무국'], dtype=object)

In [13]:
df

,연번,사용자,사용일시,사용시간,사용금액,사용장소,주소,사용방법,사용내역,집행대상
0,1,의장,2019-01-02,08:44,58500,완산정,서울 관악구 봉천로 484,신용카드,의정활동 및 의회운영 관련 업무휴대경비,의장 등 9명
1,2,의장,2019-01-02,12:36,89000,제주은갈치,서울 관악구 관악로 139,신용카드,의정활동 및 의회운영 관련 업무휴대경비,의장 등 7명
2,3,의장,2019-01-04,12:53,14000,아리차이,서울 관악구 신림동길 4,신용카드,의정활동 및 의회운영 관련 업무휴대경비,의장 등 2명
3,4,의장,2019-01-09,12:33,77000,남원추어탕,서울 관악구 쑥고개로 135,신용카드,의정활동 및 의회운영 관련 업무휴대경비,의장 등 7명
4,5,의장,2019-01-09,14:48,83300,카페베네,서울 관악구 남부순환로 2082-29,신용카드,의정활동 및 의회운영 관련 업무휴대경비,의장 등 17명
...,...,...,...,...,...,...,...,...,...,...
1999,101,의회사무국,2019-12-27,20:27,132000,시골집,서울 관악구 낙성대로 22-7,신용카드,의정업무 추진관계자 간담회,사무국장 등 10명
2000,102,의회사무국,2019-12-30,13:00,220000,만리장성,서울 관악구 관악로 146,신용카드,사무국 직원 격려,의정팀장 등 10명
2001,103,의회사무국,2019-12-30,13:08,87000,쌈마을,서울 관악구 남부순환로226길 31,신용카드,의사팀 업무추진 관련 간담회비,의사팀 직원 10명
2002,104,의회사무국,2019-12-30,14:01,174000,장보고마트,서울 관악구 남부순환로230길,신용카드,부서운영 물품(음료) 구입,"의원, 직원, 방문민원인 등"


In [14]:
# 연번, 사용시간, 사용방법, 집행대상 컬럼 삭제
df = df.drop(['연번', '사용시간', '사용방법', '집행대상'], axis=1)
df.head()

,사용자,사용일시,사용금액,사용장소,주소,사용내역
0,의장,2019-01-02,58500,완산정,서울 관악구 봉천로 484,의정활동 및 의회운영 관련 업무휴대경비
1,의장,2019-01-02,89000,제주은갈치,서울 관악구 관악로 139,의정활동 및 의회운영 관련 업무휴대경비
2,의장,2019-01-04,14000,아리차이,서울 관악구 신림동길 4,의정활동 및 의회운영 관련 업무휴대경비
3,의장,2019-01-09,77000,남원추어탕,서울 관악구 쑥고개로 135,의정활동 및 의회운영 관련 업무휴대경비
4,의장,2019-01-09,83300,카페베네,서울 관악구 남부순환로 2082-29,의정활동 및 의회운영 관련 업무휴대경비


In [15]:
# 사용일시 컬럼 날짜타입으로 변경
df['사용일시'] = pd.to_datetime(df['사용일시'])

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2004 entries, 0 to 2003
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   사용자     2004 non-null   object        
 1   사용일시    2004 non-null   datetime64[ns]
 2   사용금액    2004 non-null   int64         
 3   사용장소    1993 non-null   object        
 4   주소      1993 non-null   object        
 5   사용내역    2004 non-null   object        
dtypes: datetime64[ns](1), int64(1), object(4)
memory usage: 94.1+ KB


### 결측치 확인 및 처리

In [17]:
df.isnull().sum()

사용자      0
사용일시     0
사용금액     0
사용장소    11
주소      11
사용내역     0
dtype: int64

In [18]:
# 결측치 제거
df.dropna(inplace=True)

In [19]:
# 확인
df.isnull().sum()

사용자     0
사용일시    0
사용금액    0
사용장소    0
주소      0
사용내역    0
dtype: int64

### 사용장소 전처리

In [20]:
for i in df.index:
    if df.loc[i, '사용장소']=='GS슈퍼':
        df.loc[i, '사용장소'] = 'GS슈퍼마켓'
    elif df.loc[i, '사용장소']=='강강술래신림본동점':
        df.loc[i, '사용장소'] = '강강술래'
    elif df.loc[i, '사용장소']=='강촌민물매운탕２호점':
        df.loc[i, '사용장소'] = '강촌민물매운탕2호점'
    elif df.loc[i, '사용장소']=='고봉민김밥인':
        df.loc[i, '사용장소'] = '고봉민김밥'
    elif df.loc[i, '사용장소']=='김가네 관악구청점':
        df.loc[i, '사용장소'] = '김가네'

    
    
    elif df.loc[i, '사용장소']=='동강영양탕':
        df.loc[i, '사용장소'] = '동강염소탕'
    elif df.loc[i, '사용장소']=='맛고을칼국수＆찰보리':
        df.loc[i, '사용장소'] = '맛고을칼국수＆찰보리밥'
    elif df.loc[i, '사용장소']=='먹쉬돈나 서울대역점':
        df.loc[i, '사용장소'] = '먹쉬돈나'
    elif df.loc[i, '사용장소']=='명태어장(서울대점)':
        df.loc[i, '사용장소'] = '명태어장'
    elif df.loc[i, '사용장소']=='명품청기와감자탕미성점':
        df.loc[i, '사용장소'] = '명품청기와감자탕'
    elif df.loc[i, '사용장소']=='미가할매참품한우':
        df.loc[i, '사용장소'] = '미가할매집'
    elif df.loc[i, '사용장소']=='바우네나주곰탕　봉천':
        df.loc[i, '사용장소'] = '바우네나주곰탕'
        

        
        
    elif df.loc[i, '사용장소']=='봉이 구이대장':
        df.loc[i, '사용장소'] = '봉이구이대장'
    elif df.loc[i, '사용장소']=='사시사철흑염소.오리':
        df.loc[i, '사용장소'] = '사시사철흑염소오리'
        
        

    
    elif (df.loc[i, '사용장소']=='엔조파스타&피자') or (df.loc[i, '사용장소']=='ENZO(엔조)'):
        df.loc[i, '사용장소'] = '엔조 파스타&피자'
        
        
    elif (df.loc[i, '사용장소']=='영광수산장어1번가') or (df.loc[i, '사용장소']=='영광수산장어１번가'):
        df.loc[i, '사용장소'] = '영광수산숯불민물장어'
    
    
    elif (df.loc[i, '사용장소']=='이대감명품막창') or (df.loc[i, '사용장소']=='이대감왕족발'):
        df.loc[i, '사용장소'] = '이대감 명품막창'
    
    elif (df.loc[i, '사용장소']=='일커피　서울대입구점') or (df.loc[i, '사용장소']=='일커피 서울대입구점'):
        df.loc[i, '사용장소'] = '일커피'
                                            
    elif (df.loc[i, '사용장소']=='자연별곡 서울대입구점') or (df.loc[i, '사용장소']=='자연별곡서울대입구점'):
        df.loc[i, '사용장소'] = '자연별곡'
    
    elif (df.loc[i, '사용장소']=='작은따옴표 협동조합') or (df.loc[i, '사용장소']=='작은따옴표협동조합'):
        df.loc[i, '사용장소'] = '작은따옴표'
        
    elif (df.loc[i, '사용장소']=='커피빈') or (df.loc[i, '사용장소']=='커피빈코리아인'):
        df.loc[i, '사용장소'] = '커피빈코리아'
        
    elif (df.loc[i, '사용장소']=='콩심 서울대점') or (df.loc[i, '사용장소']=='콩심서울대점'):
        df.loc[i, '사용장소'] = '콩심'
        
        
    
    elif df.loc[i, '사용장소']=='종로계림닭도리탕':
        df.loc[i, '사용장소'] = '종로계림닭도리탕원조'
        
    elif df.loc[i, '사용장소']=='진부령　황태촌':
        df.loc[i, '사용장소'] = '진부령황태촌'    
    
    elif df.loc[i, '사용장소']=='진우리참치':
        df.loc[i, '사용장소'] = '진우리독도참치'    
        
    elif df.loc[i, '사용장소']=='카페모아관악구청점':
        df.loc[i, '사용장소'] = '카페모아'    
    
    elif df.loc[i, '사용장소']=='쿠팡㈜':
        df.loc[i, '사용장소'] = '쿠팡'       
    
    elif df.loc[i, '사용장소']=='태창약국':
        df.loc[i, '사용장소'] = '태창온누리약국'     
    
    elif df.loc[i, '사용장소']=='투썸플레이스':
        df.loc[i, '사용장소'] = '투썸서울대입구역'       
        
        
    elif (df.loc[i, '사용장소']=='난곡문성파리바게뜨') or (df.loc[i, '사용장소']=='난곡사거리 파리바게뜨'):
        df.loc[i, '사용장소'] = '파리바게뜨'
    
    elif (df.loc[i, '사용장소']=='서울대 파리바게뜨') or (df.loc[i, '사용장소']=='서울대입구역2호 파리바게뜨'):
        df.loc[i, '사용장소'] = '파리바게뜨'
    elif (df.loc[i, '사용장소']=='은천길입구 파리바게뜨') or (df.loc[i, '사용장소']=='파리바게뜨서울대입구'):
        df.loc[i, '사용장소'] = '파리바게뜨'
    elif (df.loc[i, '사용장소']=='파리바게트'):
        df.loc[i, '사용장소'] = '파리바게뜨'
    
    elif (df.loc[i, '사용장소']=='본죽 관악구청점') or (df.loc[i, '사용장소']=='본죽&본도시락'):
        df.loc[i, '사용장소'] = '본죽'
    elif (df.loc[i, '사용장소']=='본죽&비빔밥') or (df.loc[i, '사용장소']=='본죽봉천점'):
        df.loc[i, '사용장소'] = '본죽'   
    elif (df.loc[i, '사용장소']=='본죽＆비빔밥') or (df.loc[i, '사용장소']=='본죽＆비빔밥ｃａｆｅ'):
        df.loc[i, '사용장소'] = '본죽'
          
          
    elif (df.loc[i, '사용장소']=='서울대학교생협호암교수회관') or (df.loc[i, '사용장소']=='서울대학교생활협동조'):
        df.loc[i, '사용장소'] = '서울대학교'
          
          
    elif (df.loc[i, '사용장소']=='스타벅스 서울대입구역점'):
        df.loc[i, '사용장소'] = '스타벅스'
    
    elif (df.loc[i, '사용장소']=='아웃백') or (df.loc[i, '사용장소']=='아웃백스테이크'):
        df.loc[i, '사용장소'] = '아웃백스테이크하우스'
    
    elif (df.loc[i, '사용장소']=='삼백돈'):
        df.loc[i, '사용장소'] = '삼백돈돈가츠'      
    elif (df.loc[i, '사용장소']=='두부이야기봉천점'):
        df.loc[i, '사용장소'] = '두부이야기'
    elif (df.loc[i, '사용장소']=='얼큰수제비') or (df.loc[i, '사용장소']=='얼큰수제비 해물칼국수'):
        df.loc[i, '사용장소'] = '얼큰수제비해물칼국수'
        
    elif (df.loc[i, '사용장소']=='얼큰수제비') or (df.loc[i, '사용장소']=='얼큰수제비 해물칼국수'):
        df.loc[i, '사용장소'] = '얼큰수제비해물칼국수'

In [21]:
df['사용장소'].value_counts()

제주은갈치     93
시골집       45
옥천골       40
남원추어탕     33
감나무집      33
          ..
개성보쌈       1
오뚜기식당      1
카페드구띠에     1
다빈         1
풍성구이       1
Name: 사용장소, Length: 557, dtype: int64

## 사용장소별 데이터 확인하기

In [22]:
df1 = df.value_counts('사용장소').reset_index().sort_values('사용장소')
df1.reset_index(inplace=True)
df1.drop('index', axis=1, inplace=True)
df1

,사용장소,0
0,#207,2
1,(주)소명생고기,1
2,(주)송원마포갈비,1
3,(주)신세계페이먼츠,4
4,(주)신화아이푸드,1
...,...,...
552,흥부보쌈,8
553,희로애락,2
554,희전명가,1
555,히포커피,1


In [23]:
df2 = df.groupby('사용장소').sum()
df2.reset_index(inplace=True)
df2

,사용장소,사용금액
0,#207,27200
1,(주)소명생고기,100000
2,(주)송원마포갈비,45500
3,(주)신세계페이먼츠,546290
4,(주)신화아이푸드,469500
...,...,...
552,흥부보쌈,1167000
553,희로애락,508000
554,희전명가,88000
555,히포커피,10800


In [24]:
for i in range(len(df2)):
    if df2.loc[i, '사용장소']==df1.loc[i, '사용장소']:
        df2.loc[i, '방문횟수'] = df1.loc[i, 0]

In [25]:
df2.isnull().sum()

사용장소    0
사용금액    0
방문횟수    0
dtype: int64

In [26]:
# 방문횟수 int로 변경
df2['방문횟수'] = df2['방문횟수'].astype('int')
df2.head()

,사용장소,사용금액,방문횟수
0,#207,27200,2
1,(주)소명생고기,100000,1
2,(주)송원마포갈비,45500,1
3,(주)신세계페이먼츠,546290,4
4,(주)신화아이푸드,469500,1


In [27]:
df2.sort_values('방문횟수', ascending=False).head(30)

,사용장소,사용금액,방문횟수
431,제주은갈치,11714000,93
285,시골집,9381000,45
346,옥천골,6801000,40
506,파리바게뜨,1125300,33
79,남원추어탕,3146000,33
23,감나무집,6527500,33
239,산야로,1916000,31
338,오리오리,2076000,31
298,쌈마을,4350000,26
334,옛날육개장,2664000,26


In [28]:
df2 = df2[['사용장소', '방문횟수', '사용금액']]
df2

,사용장소,방문횟수,사용금액
0,#207,2,27200
1,(주)소명생고기,1,100000
2,(주)송원마포갈비,1,45500
3,(주)신세계페이먼츠,4,546290
4,(주)신화아이푸드,1,469500
...,...,...,...
552,흥부보쌈,8,1167000
553,희로애락,2,508000
554,희전명가,1,88000
555,히포커피,1,10800


### 데이터 저장하기

In [29]:
# df2.to_csv('./dataset/업무추진비_방문횟수_사용금액_557.csv', index=False)

잘못 기재된 데이터들을 확인하는 작업이 필요하다.

In [30]:
len(df2[df2['방문횟수']>3])

128

## 사용자별 데이터 확인하기

In [31]:
df['사용자'].unique()

array(['의장', '부의장', '의회운영위원장', '행정재경위원장', '보건복지위원장', '도시건설위원장', '의회사무국'],
      dtype=object)

In [32]:
df[df['사용자']=='의장'].groupby('사용장소').count().sort_values('사용자', ascending=False).head(25)

,사용자,사용일시,사용금액,주소,사용내역
사용장소,,,,,
제주은갈치,33,33,33,33,33
우리동네고기집,22,22,22,22,22
감나무집,18,18,18,18,18
생각보다맛있는집,16,16,16,16,16
남원추어탕,15,15,15,15,15
떡보의하루,11,11,11,11,11
옛날육개장,9,9,9,9,9
시골집,9,9,9,9,9
정가,7,7,7,7,7


In [33]:
df_user_1 = df[df['사용자']=='의장']['사용장소'].value_counts().reset_index()
df_user_1 = df_user_1.sort_values('index').reset_index().drop('level_0', axis=1)
df_user_1

,index,사용장소
0,감나무집,18
1,강강술래,7
2,강촌민물매운탕2호점,1
3,개성보쌈,1
4,갯벌낙지,1
...,...,...
60,파리바게뜨,3
61,풍천장어,1
62,하이보,4
63,한류관,1


In [34]:
df_user_2 = df[df['사용자']=='의장'].groupby('사용장소').sum().reset_index()

for i in df_user_1.index:
    if df_user_2.loc[i, '사용장소'] == df_user_1.loc[i, 'index']:
        df_user_2.loc[i, '방문횟수'] = df_user_1.loc[i, '사용장소']

df_user_2['방문횟수'] = df_user_2['방문횟수'].astype('int')
df_user_2 = df_user_2[['사용장소', '방문횟수', '사용금액']]

df_user_2.sort_values('방문횟수', ascending=False).head()

,사용장소,방문횟수,사용금액
50,제주은갈치,33,4435000
44,우리동네고기집,22,7319900
0,감나무집,18,3137000
24,생각보다맛있는집,16,1005000
8,남원추어탕,15,1610000


### 함수 만들어보기

In [35]:
def df_user(user):
    df_user_1 = df[df['사용자']==user]['사용장소'].value_counts().reset_index()
    df_user_1 = df_user_1.sort_values('index').reset_index().drop('level_0', axis=1)
    
    df_user_2 = df[df['사용자']==user].groupby('사용장소').sum().reset_index()

    for i in df_user_1.index:
        if df_user_2.loc[i, '사용장소'] == df_user_1.loc[i, 'index']:
            df_user_2.loc[i, '방문횟수'] = df_user_1.loc[i, '사용장소']

    df_user_2['방문횟수'] = df_user_2['방문횟수'].astype('int')
    df_user_2 = df_user_2[['사용장소', '방문횟수', '사용금액']]

    return df_user_2.sort_values('방문횟수', ascending=False).head()

In [36]:
df_user('의장')

,사용장소,방문횟수,사용금액
50,제주은갈치,33,4435000
44,우리동네고기집,22,7319900
0,감나무집,18,3137000
24,생각보다맛있는집,16,1005000
8,남원추어탕,15,1610000


In [37]:
df_user('부의장')

,사용장소,방문횟수,사용금액
80,은성숯불갈비,19,1867100
20,당곡기사식당,11,846000
6,고창복의낙지세상,10,913000
7,구수옥설렁탕,9,586000
42,본죽,7,154000


In [38]:
df_user('의회운영위원장')

,사용장소,방문횟수,사용금액
17,다리목,11,2560000
38,북경반점,7,763000
61,장터순대국,5,743000
26,명품횡성한우가,5,602000
9,김밥,5,1021000


In [39]:
df_user('행정재경위원장')

,사용장소,방문횟수,사용금액
87,통큰대박집,10,2435000
74,전진식당,10,314000
8,거목쟁반짜장,8,136500
85,콩심,8,136000
15,금비,6,1749000


In [40]:
df_user('보건복지위원장')

,사용장소,방문횟수,사용금액
34,옥천골,37,6065000
49,제주은갈치,20,3097000
2,감나무집,6,1026000
47,정육점식당,5,227000
20,목포부부아구찜,5,580000


In [41]:
df_user('도시건설위원장')

,사용장소,방문횟수,사용금액
98,파리바게뜨,8,281400
109,흥부보쌈,5,666000
45,사랑방,5,951000
25,만리장성,5,384500
1,감나무집,5,1684500


In [42]:
df_user('의회사무국')

,사용장소,방문횟수,사용금액
112,시골집,33,6855000
173,제주은갈치,29,3661000
88,산야로,29,1867000
133,오리오리,29,1949000
162,장보고마트,25,6522210


## 위치 데이터 추가

In [43]:
pd.read_csv('./dataset/관악구 업무추진비 위치.csv')

,사용장소,주소,사용금액,lat,lng,거리
0,#207,서울 관악구 원신길 128 #207,27200,37.506248,126.913952,4559.495385
1,(주)소명생고기,서울 금천구 시흥대로 264 (주)소명생고기,100000,37.457203,126.899928,5111.949204
2,(주)송원마포갈비,서울 영등포구 도림로147길 2-1 (주)송원마포갈비,45500,37.516644,126.889236,6969.211685
3,(주)신세계페이먼츠,서울 중구 남대문시장10길 2 (주)신세계페이먼츠,546290,37.560333,126.982963,9555.694239
4,(주)신화아이푸드,충청북도 청주시 서원구 1순환로 886 (주)신화아이푸드,469500,36.616615,127.469836,106263.797527
...,...,...,...,...,...,...
586,흥부보쌈,서울 관악구 신림로59길 15-13 흥부보쌈,1167000,37.482655,126.929366,2019.981634
587,희로애락,서울 관악구 봉천로 465 희로애락,508000,37.482469,126.951614,486.587335
588,희전명가,서울 관악구 남부순환로180길 6 희전명가,88000,37.485211,126.931504,1935.672971
589,히포커피,서울 관악구 신림로23길 21 히포커피,10800,37.469969,126.934968,1717.771051
